# Wikidata5M passage, entity-link, and local-graph inspection

This notebook inspects the first 20 `wikidata5m_text.txt` records without building the production preprocessing pipeline. It answers three questions: **what aliases match the text, which matches are plausible entity mentions, and do the mentioned entities have direct or two-hop connections in Wikidata5M?**

> Important limitation: the text corpus has no Wikipedia hyperlink annotations. Exact alias matching therefore produces candidates, not ground-truth entity links. Common aliases such as *on*, *English*, or *first* can map to entities. The notebook deliberately retains **all raw alias hits** for auditing, then applies a conservative, documented resolver before using mentions as graph seeds. This exposes rather than hides the main weakness of alias-only linking.

## Efficient sample-only strategy

Loading 4.8 million alias rows or 20.6 million triples into Python objects would be wasteful. Instead, this notebook:

1. reads only the first 20 passages;
2. builds the set of token-bounded substrings that can occur in those passages;
3. streams the entity-alias file once, retaining only aliases present in that set;
4. resolves a conservative, non-overlapping subset of matches;
5. streams the graph files once, retaining every triple incident on one of those resolved entities; and
6. joins incident edges to find direct and two-hop connections between entities in the same passage.

The two-hop join treats edges as undirected **only for connectivity discovery**. The displayed and stored triples retain their original Wikidata direction and relation ID.

In [10]:
from __future__ import annotations

from collections import defaultdict
from html import escape
from itertools import combinations
from pathlib import Path
import math
import re

from IPython.display import HTML, Markdown, display
from tqdm.auto import tqdm

def render_table(rows, max_rows=None):
    if not rows:
        return HTML('<em>No rows.</em>')
    visible = rows if max_rows is None else rows[:max_rows]
    headers = list(visible[0])
    head = ''.join(f'<th style="text-align:left;padding:4px 8px">{escape(str(h))}</th>' for h in headers)
    body = ''.join(
        '<tr>' + ''.join(f'<td style="vertical-align:top;padding:4px 8px">{escape(str(row.get(h, "")))}</td>' for h in headers) + '</tr>'
        for row in visible
    )
    suffix = f'<p><em>Showing {len(visible):,} of {len(rows):,} rows.</em></p>' if len(visible) < len(rows) else ''
    return HTML(f'<div style="overflow-x:auto"><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>{suffix}')

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'datasets' / 'Wikidata5M').is_dir():
            return candidate
    raise FileNotFoundError('Could not find datasets/Wikidata5M above the notebook working directory.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA_ROOT = REPO_ROOT / 'datasets' / 'Wikidata5M'
TEXT_PATH = DATA_ROOT / 'wikidata5m_text.txt'
ENTITY_ALIAS_PATH = DATA_ROOT / 'wikidata5m_alias' / 'wikidata5m_entity.txt'
RELATION_ALIAS_PATH = DATA_ROOT / 'wikidata5m_alias' / 'wikidata5m_relation.txt'
TRIPLE_PATHS = [
    DATA_ROOT / 'wikidata5m_transductive' / f'wikidata5m_transductive_{split}.txt'
    for split in ('train', 'valid', 'test')
]

NUM_PASSAGES = 20
MAX_ALIAS_TOKENS = None  # None checks every token-bounded substring in these 20 short passages.
MAX_STORED_TWO_HOP_PATHS = 2_000  # Safety cap for hub-heavy samples; total counts remain available.
EGO_EDGES_PER_ENTITY = 2  # Fallback visualization if no connecting path exists.
MAX_GRAPH_DISPLAY_EDGES = 80  # Records keep all connecting edges; SVGs stay readable.
MAX_RAW_ALIAS_ROWS_DISPLAY = 50  # Every hit remains in raw_alias_mentions.

for path in [TEXT_PATH, ENTITY_ALIAS_PATH, RELATION_ALIAS_PATH, *TRIPLE_PATHS]:
    assert path.is_file(), f'Missing required file: {path}'

DATA_ROOT


/Users/diyalakhani/Documents/Diya/ucla/thesis/KGReaLM/oreo-decoder/baseline/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/Users/diyalakhani/Documents/Diya/ucla/thesis/KGReaLM/datasets/Wikidata5M')

In [11]:
passages = []
with TEXT_PATH.open(encoding='utf-8') as handle:
    for index, line in enumerate(handle):
        if index >= NUM_PASSAGES:
            break
        qid, text = line.rstrip('\n').split('\t', 1)
        passages.append({'index': index, 'passage_qid': qid, 'text': text})

display(render_table(
    [{'index': p['index'], 'passage_qid': p['passage_qid'], 'characters': len(p['text']),
      'preview': p['text'][:120] + ('…' if len(p['text']) > 120 else '')} for p in passages]
))


index,passage_qid,characters,preview
0,Q7594088,1909,"St Magnus the Martyr, London Bridge is a Church of England church and parish within the City of London. The church, whic…"
1,Q1640233,361,The Monumentum Ancyranum (Latin 'Monument of Ancyra') or Temple of Augustus and Rome in Ancyra is an Augusteum in Ankara…
2,Q4761308,1995,"Andy Selway is an English musician, most well known for his work with KMFDM.Selway spent his early years in Luton, Engla…"
3,Q51899,139,Carapelle (Foggiano: Crapell) is a town and comune belonging to the Province of Foggia and situated in the Apulia region…
4,Q8016489,65,William P. Perry is an American composer and television producer.
5,Q8047570,512,YamaArashi (stylized as YAMAARASHI (山嵐)) is a six-member Japanese rock group based in Kanagawa Prefecture. They use a bl…
6,Q7084419,2169,"Old Main, University of Arizona, originally known as the University of Arizona, School of Agriculture building, was the …"
7,Q5495280,164,"Fred Goodwins (26 February 1891 – April 1923) was an English actor, film director and screenwriter of the silent era. He…"
8,Q6327540,146,"KCYA (97.7 FM) is a radio station licensed to Rolling Hills, Wyoming, United States. The station is currently owned by C…"
9,Q4923500,301,The Blackville and Alston Railroad was a South Carolina railroad company chartered in the latter half of the 19th centur…


## 1. Retrieve every exact alias hit in the sample

We normalize case and repeated whitespace but require exact token-bounded text. Each raw record retains every candidate QID for the matching alias. Character offsets are half-open: `[start, end)`.

In [12]:
TOKEN_RE = re.compile(r"\w+(?:[’'\-]\w+)*", re.UNICODE)

def normalize_alias(value: str) -> str:
    return ' '.join(value.casefold().split())

candidate_occurrences = defaultdict(list)
for passage in passages:
    tokens = list(TOKEN_RE.finditer(passage['text']))
    max_width = len(tokens) if MAX_ALIAS_TOKENS is None else min(MAX_ALIAS_TOKENS, len(tokens))
    for start_token in range(len(tokens)):
        for end_token in range(start_token, min(len(tokens), start_token + max_width)):
            start = tokens[start_token].start()
            end = tokens[end_token].end()
            key = normalize_alias(passage['text'][start:end])
            candidate_occurrences[key].append((passage['index'], start, end))

alias_to_qids = defaultdict(set)
entity_labels = {}
passage_qids = {p['passage_qid'] for p in passages}

with ENTITY_ALIAS_PATH.open(encoding='utf-8') as handle:
    for line in tqdm(handle, desc='Scanning entity aliases', unit=' rows'):
        fields = line.rstrip('\n').split('\t')
        if len(fields) < 2:
            continue
        qid, aliases = fields[0], fields[1:]
        if qid in passage_qids:
            entity_labels.setdefault(qid, aliases[0])
        matched = False
        for alias in aliases:
            key = normalize_alias(alias)
            if key in candidate_occurrences:
                alias_to_qids[key].add(qid)
                matched = True
        if matched:
            entity_labels.setdefault(qid, aliases[0])

print(f'{len(candidate_occurrences):,} candidate substrings from {len(passages)} passages')
print(f'{len(alias_to_qids):,} distinct aliases matched at least one Wikidata5M entity')
print(f'{sum(len(qids) for qids in alias_to_qids.values()):,} alias → entity candidate links retained')


Scanning entity aliases: 4813491 rows [00:13, 348342.62 rows/s]

214,577 candidate substrings from 20 passages
953 distinct aliases matched at least one Wikidata5M entity
1,202 alias → entity candidate links retained


## 2. Preserve raw candidates, then derive conservative graph seeds

The raw list is the answer to “what does the alias dictionary recognize?” It is intentionally noisy. The resolved list is for graph inspection and uses these rules:

- if the passage's known subject QID is among an alias's candidates, choose it;
- otherwise require exactly one candidate QID;
- require a name-like surface (capitalized/acronym or a meaningful multiword phrase); and
- greedily retain non-overlapping spans, preferring subject matches and then longer spans.

This is a high-precision heuristic, not a production entity linker. Lowercase concepts and ambiguous names can remain unresolved even when they are real mentions.

In [13]:
STOP_ALIASES = {
    'a', 'an', 'the', 'and', 'or', 'but', 'of', 'to', 'in', 'on', 'at', 'for', 'from', 'by',
    'with', 'as', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'it', 'its', 'he', 'she',
    'they', 'them', 'his', 'her', 'their', 'this', 'that', 'these', 'those', 'who', 'which',
    'what', 'when', 'where', 'why', 'how', 'not', 'no', 'yes', 'first', 'last', 'new', 'known',
    'use', 'uses', 'used', 'one', 'two', 'three', 'most', 'more', 'also', 'other', 'year', 'years'
}

def looks_name_like(surface: str, is_subject_match: bool) -> bool:
    if is_subject_match:
        return True
    words = TOKEN_RE.findall(surface)
    normalized = normalize_alias(surface)
    if not words or normalized in STOP_ALIASES or not any(ch.isalpha() for ch in surface):
        return False
    if len(words) == 1:
        token = words[0]
        return len(token) >= 3 and (token.isupper() or token[0].isupper())
    content_words = words[1:] if words[0].casefold() in STOP_ALIASES else words
    return any(word.isupper() or word[0].isupper() for word in content_words)

def spans_overlap(left, right) -> bool:
    return left['start'] < right['end'] and right['start'] < left['end']

inspection_records = []
for passage in passages:
    span_candidates = {}
    for alias, occurrences in candidate_occurrences.items():
        qids = alias_to_qids.get(alias)
        if not qids:
            continue
        for passage_index, start, end in occurrences:
            if passage_index != passage['index']:
                continue
            key = (start, end)
            entry = span_candidates.setdefault(key, {'aliases': set(), 'candidate_qids': set()})
            entry['aliases'].add(alias)
            entry['candidate_qids'].update(qids)

    raw_mentions = []
    eligible = []
    for (start, end), values in span_candidates.items():
        candidates = sorted(values['candidate_qids'])
        subject_match = passage['passage_qid'] in candidates
        resolved_qid = passage['passage_qid'] if subject_match else (candidates[0] if len(candidates) == 1 else None)
        raw = {
            'start': start, 'end': end, 'surface': passage['text'][start:end],
            'normalized_aliases': sorted(values['aliases']), 'candidate_qids': candidates,
            'is_ambiguous': len(candidates) > 1, 'resolved_qid': resolved_qid,
            'is_subject_match': subject_match,
        }
        raw_mentions.append(raw)
        if resolved_qid is not None and looks_name_like(raw['surface'], subject_match):
            eligible.append({
                'start': start, 'end': end, 'surface': raw['surface'], 'entity_id': resolved_qid,
                'source': 'subject_alias' if subject_match else 'unique_name_like_alias',
                'candidate_qids': candidates,
            })

    selected = []
    for mention in sorted(
        eligible,
        key=lambda item: (item['source'] != 'subject_alias', -(item['end'] - item['start']), item['start']),
    ):
        if not any(spans_overlap(mention, existing) for existing in selected):
            selected.append(mention)
    selected.sort(key=lambda item: (item['start'], item['end']))

    entity_spans = defaultdict(list)
    for mention in selected:
        entity_spans[mention['entity_id']].append([mention['start'], mention['end']])

    inspection_records.append({
        'passage_index': passage['index'], 'passage_qid': passage['passage_qid'], 'text': passage['text'],
        'raw_alias_mentions': sorted(raw_mentions, key=lambda item: (item['start'], item['end'])),
        'mentions': selected, 'entity_spans': dict(entity_spans),
    })

mention_summary = [{
    'passage': record['passage_index'], 'subject': record['passage_qid'],
    'raw alias spans': len(record['raw_alias_mentions']),
    'ambiguous spans': sum(item['is_ambiguous'] for item in record['raw_alias_mentions']),
    'resolved mentions': len(record['mentions']), 'resolved entities': len(record['entity_spans']),
} for record in inspection_records]
display(render_table(mention_summary))


passage,subject,raw alias spans,ambiguous spans,resolved mentions,resolved entities
0,Q7594088,344,161,33,28
1,Q1640233,55,24,11,10
2,Q4761308,310,142,50,44
3,Q51899,23,11,4,4
4,Q8016489,12,6,2,2
5,Q8047570,75,32,14,9
6,Q7084419,369,126,44,35
7,Q5495280,29,10,4,4
8,Q6327540,28,10,7,7
9,Q4923500,50,22,5,4


## 3. Retrieve incident triples and build passage-local graphs

Every resolved entity is a graph seed. The known passage subject is also included as a context seed even if its name is not literally present in the passage. One sequential scan retains **all triples incident on any seed**.

A full depth-two DFS around every entity can explode around hubs and mostly returns unrelated neighbors. For this inspection, we instead keep direct edges between passage entities and two-edge paths whose endpoints are passage entities. That is the useful structure for asking whether a passage contains relational supervision. A tiny ego graph is displayed only when no connecting structure exists.

In [14]:
graph_seeds = set()
for record in inspection_records:
    graph_seeds.add(record['passage_qid'])
    graph_seeds.update(record['entity_spans'])

incident_edges_by_entity = defaultdict(list)
for triple_path in TRIPLE_PATHS:
    with triple_path.open(encoding='utf-8') as handle:
        for line in tqdm(handle, desc=f'Scanning {triple_path.stem}', unit=' triples'):
            source, relation, target = line.rstrip('\n').split('\t')
            edge = (source, relation, target)
            if source in graph_seeds:
                incident_edges_by_entity[source].append(edge)
            if target in graph_seeds and target != source:
                incident_edges_by_entity[target].append(edge)

print(f'{len(graph_seeds):,} unique graph seeds')
print(f'{sum(len(edges) for edges in incident_edges_by_entity.values()):,} retained seed–triple incidences')


Scanning wikidata5m_transductive_train: 20614279 triples [00:09, 2112692.85 triples/s]
Scanning wikidata5m_transductive_valid: 5163 triples [00:00, 1875395.48 triples/s]
Scanning wikidata5m_transductive_test: 5133 triples [00:00, 1802374.42 triples/s]

228 unique graph seeds
1,504,402 retained seed–triple incidences


In [15]:
def other_endpoint(edge, entity_id):
    source, _, target = edge
    if source == entity_id:
        return target
    if target == entity_id:
        return source
    raise ValueError(f'{entity_id} is not incident on {edge}')

for record in inspection_records:
    passage_entities = set(record['entity_spans']) | {record['passage_qid']}
    direct_edges = set()
    connector_to_seed_edges = defaultdict(lambda: defaultdict(set))

    for seed in passage_entities:
        for edge in incident_edges_by_entity.get(seed, []):
            neighbor = other_endpoint(edge, seed)
            if neighbor in passage_entities:
                direct_edges.add(edge)
            else:
                connector_to_seed_edges[neighbor][seed].add(edge)

    two_hop_paths = []
    two_hop_path_count = 0
    path_edges = set()
    for connector, seed_edges in connector_to_seed_edges.items():
        seeds = sorted(seed_edges)
        if len(seeds) < 2:
            continue
        for left_seed, right_seed in combinations(seeds, 2):
            for left_edge in sorted(seed_edges[left_seed]):
                for right_edge in sorted(seed_edges[right_seed]):
                    two_hop_path_count += 1
                    path_edges.update((left_edge, right_edge))
                    if len(two_hop_paths) < MAX_STORED_TWO_HOP_PATHS:
                        two_hop_paths.append({
                            'left_entity': left_seed, 'connector': connector, 'right_entity': right_seed,
                            'left_edge': left_edge, 'right_edge': right_edge,
                        })

    connecting_edges = set(direct_edges) | path_edges
    display_edges = set(connecting_edges)
    used_ego_fallback = False
    if not display_edges:
        used_ego_fallback = True
        for seed in sorted(passage_entities):
            display_edges.update(incident_edges_by_entity.get(seed, [])[:EGO_EDGES_PER_ENTITY])

    record['graph'] = {
        'mentioned_entity_ids': sorted(record['entity_spans']),
        'passage_entity_ids': sorted(passage_entities),
        'incident_triples_by_entity': {
            qid: list(incident_edges_by_entity.get(qid, [])) for qid in sorted(passage_entities)
        },
        'incident_edge_counts': {
            qid: len(incident_edges_by_entity.get(qid, [])) for qid in sorted(passage_entities)
        },
        'direct_edges': sorted(direct_edges),
        'two_hop_paths': two_hop_paths,
        'two_hop_path_count': two_hop_path_count,
        'two_hop_paths_truncated': two_hop_path_count > len(two_hop_paths),
        'display_edges': sorted(display_edges),
        'used_ego_fallback': used_ego_fallback,
    }


## 4. Fetch readable labels only for nodes and relations we display

The relation file is tiny and can be loaded directly. The entity file is streamed a second time, but only labels for the small set of local-graph nodes are retained.

In [16]:
relation_labels = {}
with RELATION_ALIAS_PATH.open(encoding='utf-8') as handle:
    for line in handle:
        fields = line.rstrip('\n').split('\t')
        if len(fields) >= 2:
            relation_labels[fields[0]] = fields[1]

needed_entity_labels = set()
for record in inspection_records:
    needed_entity_labels.update(record['graph']['passage_entity_ids'])
    for source, _, target in record['graph']['display_edges']:
        needed_entity_labels.update((source, target))

missing_labels = needed_entity_labels - entity_labels.keys()
if missing_labels:
    with ENTITY_ALIAS_PATH.open(encoding='utf-8') as handle:
        for line in tqdm(handle, desc='Fetching local graph labels', unit=' rows'):
            qid, separator, aliases = line.rstrip('\n').partition('\t')
            if qid in missing_labels and separator:
                entity_labels[qid] = aliases.split('\t', 1)[0]

def entity_label(qid):
    return entity_labels.get(qid, qid)

def relation_label(pid):
    return relation_labels.get(pid, pid)


Fetching local graph labels: 4813491 rows [00:02, 1698201.45 rows/s]


## 5. Corpus-level result summary

“Two-hop paths” counts pairs of original directed triples joined through one intermediate node; direction is not required during discovery. Inspect the actual arrows and relation labels before treating a connection as useful reasoning supervision.

In [17]:
result_summary = [{
    'passage': record['passage_index'],
    'subject': f"{entity_label(record['passage_qid'])} ({record['passage_qid']})",
    'raw alias spans': len(record['raw_alias_mentions']),
    'resolved mentions': len(record['mentions']),
    'resolved entities': len(record['entity_spans']),
    'direct triples': len(record['graph']['direct_edges']),
    'two-hop paths': record['graph']['two_hop_path_count'],
    'fallback ego graph': record['graph']['used_ego_fallback'],
} for record in inspection_records]
display(render_table(result_summary))
display(Markdown(
    f"**{sum(row['direct triples'] > 0 for row in result_summary)}/{len(result_summary)}** passages contain a direct edge and "
    f"**{sum(row['two-hop paths'] > 0 for row in result_summary)}/{len(result_summary)}** contain at least one two-hop connection "
    "between passage-local entities under this conservative resolver."
))


passage,subject,raw alias spans,resolved mentions,resolved entities,direct triples,two-hop paths,fallback ego graph
0,St Magnus the Martyr (Q7594088),344,33,28,17,266,False
1,Monumentum Ancyranum (Q1640233),55,11,10,6,907,False
2,Andy Selway (Q4761308),310,50,44,39,33627,False
3,Carapelle (Q51899),23,4,4,5,47,False
4,William P. Perry (Q8016489),12,2,2,0,0,True
5,yamaarashi (Q8047570),75,14,9,0,13,False
6,"Old Main, University of Arizona (Q7084419)",369,44,35,10,10976,False
7,Fred Goodwins (Q5495280),29,4,4,0,0,True
8,KCYA (Q6327540),28,7,7,5,108,False
9,blackville and alston railroad (Q4923500),50,5,4,0,6,False


**14/20** passages contain a direct edge and **17/20** contain at least one two-hop connection between passage-local entities under this conservative resolver.

In [18]:
def highlighted_passage(record):
    chunks = []
    cursor = 0
    for mention in record['mentions']:
        chunks.append(escape(record['text'][cursor:mention['start']]))
        tooltip = escape(f"{entity_label(mention['entity_id'])} | {mention['entity_id']} | {mention['source']}")
        chunks.append(f"<mark title='{tooltip}'>{escape(record['text'][mention['start']:mention['end']])}</mark>")
        cursor = mention['end']
    chunks.append(escape(record['text'][cursor:]))
    return "<div style='line-height:1.75; max-width:1000px'>" + ''.join(chunks) + '</div>'

def edge_text(edge):
    source, relation, target = edge
    return f"{entity_label(source)} ({source}) —[{relation_label(relation)} ({relation})]→ {entity_label(target)} ({target})"

def draw_local_graph(record):
    all_edges = record['graph']['display_edges']
    edges = all_edges[:MAX_GRAPH_DISPLAY_EDGES]
    if not edges:
        display(Markdown('*No incident triples found for the passage entities.*'))
        return
    nodes = sorted({node for source, _, target in edges for node in (source, target)})
    edge_relations = defaultdict(list)
    for source, relation, target in edges:
        edge_relations[(source, target)].append(relation_label(relation))

    passage_entities = set(record['graph']['passage_entity_ids'])
    mentioned_entities = set(record['graph']['mentioned_entity_ids'])
    width, height = 1000, 700
    center_x, center_y = width / 2, height / 2
    radius = min(width, height) * 0.37
    positions = {
        node: (center_x + radius * math.cos(2 * math.pi * i / len(nodes) - math.pi / 2),
               center_y + radius * math.sin(2 * math.pi * i / len(nodes) - math.pi / 2))
        for i, node in enumerate(nodes)
    }
    marker_id = f"arrow-{record['passage_index']}"
    svg = [
        f'<svg viewBox="0 0 {width} {height}" style="width:100%;max-width:1100px;border:1px solid #ddd;background:#fff">',
        f'<defs><marker id="{marker_id}" markerWidth="10" markerHeight="7" refX="9" refY="3.5" orient="auto"><polygon points="0 0, 10 3.5, 0 7" fill="#666"/></marker></defs>',
    ]
    for (source, target), relations in edge_relations.items():
        x1, y1 = positions[source]
        x2, y2 = positions[target]
        dx, dy = x2 - x1, y2 - y1
        distance = max(math.hypot(dx, dy), 1)
        ux, uy = dx / distance, dy / distance
        x1, y1, x2, y2 = x1 + ux * 36, y1 + uy * 36, x2 - ux * 42, y2 - uy * 42
        relation_text = ', '.join(dict.fromkeys(relations[:3])) + ('…' if len(relations) > 3 else '')
        svg.append(f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" stroke="#666" stroke-width="1.4" marker-end="url(#{marker_id})"/>')
        svg.append(f'<text x="{(x1+x2)/2:.1f}" y="{(y1+y2)/2-5:.1f}" text-anchor="middle" font-size="10" fill="#333">{escape(relation_text[:45])}</text>')
    for node in nodes:
        x, y = positions[node]
        color = '#ef6c5b' if node in mentioned_entities else '#f4b942' if node in passage_entities else '#8ab6d6'
        svg.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="34" fill="{color}" stroke="#444"/>')
        svg.append(f'<text x="{x:.1f}" y="{y-3:.1f}" text-anchor="middle" font-size="10" font-weight="600">{escape(entity_label(node)[:22])}</text>')
        svg.append(f'<text x="{x:.1f}" y="{y+12:.1f}" text-anchor="middle" font-size="9">{escape(node)}</text>')
    svg.append('</svg>')
    title = 'Fallback ego neighborhood' if record['graph']['used_ego_fallback'] else 'Direct and two-hop connecting graph'
    truncation = f' (showing {len(edges):,} of {len(all_edges):,} edges)' if len(edges) < len(all_edges) else ''
    display(Markdown(f'### {title}{truncation}'))
    display(HTML(''.join(svg)))

def inspect_passage(index):
    record = inspection_records[index]
    graph = record['graph']
    display(Markdown(f"## Passage {index}: {entity_label(record['passage_qid'])} (`{record['passage_qid']}`)"))
    display(HTML(highlighted_passage(record)))
    display(Markdown(
        f"**Raw alias spans:** {len(record['raw_alias_mentions']):,} &nbsp; | &nbsp; "
        f"**resolved mentions:** {len(record['mentions']):,} &nbsp; | &nbsp; "
        f"**direct triples:** {len(graph['direct_edges']):,} &nbsp; | &nbsp; "
        f"**two-hop paths:** {graph['two_hop_path_count']:,}"
    ))

    mention_rows = []
    for qid, spans in record['entity_spans'].items():
        surfaces = [record['text'][start:end] for start, end in spans]
        mention_rows.append({
            'entity ID': qid, 'label': entity_label(qid), 'surface(s)': ' | '.join(surfaces),
            'character spans': spans, 'incident triples': graph['incident_edge_counts'].get(qid, 0),
        })
    if record['passage_qid'] not in record['entity_spans']:
        mention_rows.insert(0, {
            'entity ID': record['passage_qid'], 'label': entity_label(record['passage_qid']),
            'surface(s)': '(passage subject; not resolved in text)', 'character spans': [],
            'incident triples': graph['incident_edge_counts'].get(record['passage_qid'], 0),
        })
    display(render_table(mention_rows))
    raw_rows = [{
        'surface': item['surface'], 'span': [item['start'], item['end']],
        'candidate QIDs': item['candidate_qids'], 'ambiguous': item['is_ambiguous'],
        'selected entity': item['resolved_qid'] or '',
    } for item in record['raw_alias_mentions']]
    display(Markdown(f"### Raw alias candidates (showing up to {MAX_RAW_ALIAS_ROWS_DISPLAY})"))
    display(render_table(raw_rows, max_rows=MAX_RAW_ALIAS_ROWS_DISPLAY))

    if graph['direct_edges']:
        display(Markdown('### Direct triples'))
        display(render_table([{'triple': edge_text(edge)} for edge in graph['direct_edges']]))
    if graph['two_hop_paths']:
        display(Markdown('### Two-hop paths' + (' (stored sample; count was capped)' if graph['two_hop_paths_truncated'] else '')))
        path_rows = []
        for path in graph['two_hop_paths']:
            path_rows.append({
                'left entity': f"{entity_label(path['left_entity'])} ({path['left_entity']})",
                'first original triple': edge_text(path['left_edge']),
                'connector': f"{entity_label(path['connector'])} ({path['connector']})",
                'second original triple': edge_text(path['right_edge']),
                'right entity': f"{entity_label(path['right_entity'])} ({path['right_entity']})",
            })
        display(render_table(path_rows, max_rows=100))
    if graph['used_ego_fallback']:
        display(Markdown('*No direct/two-hop connection was found; the plot shows up to two incident triples per entity instead.*'))
    draw_local_graph(record)


## 6. Inspect every sample

Run the following cell to render all 20 passages. For a compact iteration during development, replace the loop with `inspect_passage(0)`. The complete Python object is `inspection_records`; raw candidates for passage `i` are in `inspection_records[i]['raw_alias_mentions']`, and every retained incident triple is in `inspection_records[i]['graph']['incident_triples_by_entity']`.

In [19]:
for passage_index in range(len(inspection_records)):
    inspect_passage(passage_index)


## Passage 0: St Magnus the Martyr (`Q7594088`)

**Raw alias spans:** 344 &nbsp; | &nbsp; **resolved mentions:** 33 &nbsp; | &nbsp; **direct triples:** 17 &nbsp; | &nbsp; **two-hop paths:** 266

entity ID,label,surface(s),character spans,incident triples
Q7594088,St Magnus the Martyr,St Magnus the Martyr,"[[0, 20]]",6
Q130206,Loddon Bridge,London Bridge | London Bridge | London Bridge,"[[22, 35], [494, 507], [1255, 1268]]",13
Q82708,Church of England,Church of England | the Church of England,"[[41, 58], [402, 423]]",288
Q23311,city bars,the City of London | the City,"[[84, 102], [516, 524]]",447
Q1503695,lower thames street,Lower Thames Street,"[[136, 155]]",4
Q607700,Monument to the Great Fire of London,Monument to the Great Fire of London,"[[165, 201]]",9
Q14946379,Anglican Diocese of London,Diocese of London,"[[218, 235]]",169
Q866112,Bishop of Fulham,Bishop of Fulham,"[[271, 287]]",4
Q570600,grade b listed,Grade I listed building,"[[297, 320]]",109
Q1056391,Principal of the school,Rector,"[[358, 364]]",139


### Raw alias candidates (showing up to 50)

surface,span,candidate QIDs,ambiguous,selected entity
St,"[0, 2]","['Q1039', 'Q188725', 'Q43115', 'Q855463']",True,
St Magnus the Martyr,"[0, 20]",['Q7594088'],False,Q7594088
Magnus,"[3, 9]","['Q125414', 'Q72859']",True,
Magnus the Martyr,"[3, 20]",['Q522317'],False,Q522317
the,"[10, 13]","['Q21121474', 'Q3312799']",True,
Martyr,"[14, 20]","['Q107013', 'Q6498826']",True,
London,"[22, 28]",['Q84'],False,Q84
London Bridge,"[22, 35]",['Q130206'],False,Q130206
Bridge,"[29, 35]","['Q12280', 'Q188966']",True,
is,"[36, 38]","['Q189', 'Q207509', 'Q2429253', 'Q5974632']",True,


...

### Direct triples

triple
virgin assumed (Q162691) —[religion (P140)]→ catholisism (Q9592)
bougainville (autonomous region) (Q18826) —[country (P17)]→ pua pua new guniea (Q691)
"our lady of the assumption cathedral, buka (Q23730330) —[country (P17)]→ pua pua new guniea (Q691)"
"buka, pg-nsa (Q2482800) —[country (P17)]→ pua pua new guniea (Q691)"
pua pua new guniea (Q691) —[contains administrative territorial entity (P150)]→ bougainville (autonomous region) (Q18826)
pua pua new guniea (Q691) —[country (P17)]→ pua pua new guniea (Q691)
pua pua new guniea (Q691) —[continent (P30)]→ oceania portal (Q538)


### Two-hop paths

left entity,first original triple,connector,second original triple,right entity
كاتدرائية (Q2977),"dsouzaron/sacred heart cathedral, lahore (Q3695615) —[instance of (P31)]→ كاتدرائية (Q2977)","dsouzaron/sacred heart cathedral, lahore (Q3695615)","dsouzaron/sacred heart cathedral, lahore (Q3695615) —[religion (P140)]→ catholisism (Q9592)",catholisism (Q9592)
cologne use (Q49374),cologne use (Q49374) —[used by (P1535)]→ latin-rite catholics (Q597526),latin-rite catholics (Q597526),latin-rite catholics (Q597526) —[part of (P361)]→ catholisism (Q9592),catholisism (Q9592)
cologne use (Q49374),cologne use (Q49374) —[used by (P1535)]→ latin-rite catholics (Q597526),latin-rite catholics (Q597526),catholisism (Q9592) —[has part (P527)]→ latin-rite catholics (Q597526),catholisism (Q9592)
cologne use (Q49374),latin-rite catholics (Q597526) —[uses (P2283)]→ cologne use (Q49374),latin-rite catholics (Q597526),latin-rite catholics (Q597526) —[part of (P361)]→ catholisism (Q9592),catholisism (Q9592)
cologne use (Q49374),latin-rite catholics (Q597526) —[uses (P2283)]→ cologne use (Q49374),latin-rite catholics (Q597526),catholisism (Q9592) —[has part (P527)]→ latin-rite catholics (Q597526),catholisism (Q9592)
كاتدرائية (Q2977),Cathedral of Palencia (Q2942521) —[instance of (P31)]→ كاتدرائية (Q2977),Cathedral of Palencia (Q2942521),Cathedral of Palencia (Q2942521) —[religion (P140)]→ catholisism (Q9592),catholisism (Q9592)
كاتدرائية (Q2977),"Ascension Cathedral, Satu Mare (Q667991) —[instance of (P31)]→ كاتدرائية (Q2977)","Ascension Cathedral, Satu Mare (Q667991)","Ascension Cathedral, Satu Mare (Q667991) —[religion (P140)]→ catholisism (Q9592)",catholisism (Q9592)
كاتدرائية (Q2977),cathedral of our lady (antwerp) (Q5901) —[instance of (P31)]→ كاتدرائية (Q2977),cathedral of our lady (antwerp) (Q5901),cathedral of our lady (antwerp) (Q5901) —[owned by (P127)]→ catholisism (Q9592),catholisism (Q9592)
pua pua new guniea (Q691),ezekiel anisi (Q28037442) —[country of citizenship (P27)]→ pua pua new guniea (Q691),ezekiel anisi (Q28037442),ezekiel anisi (Q28037442) —[religion (P140)]→ catholisism (Q9592),catholisism (Q9592)
كاتدرائية (Q2977),orvieto cathedral (Q10924) —[instance of (P31)]→ كاتدرائية (Q2977),orvieto cathedral (Q10924),orvieto cathedral (Q10924) —[religion (P140)]→ catholisism (Q9592),catholisism (Q9592)


### Direct and two-hop connecting graph (showing 80 of 222 edges)

## Future production storage (not implemented here)

Yes—converting the text files into indexed stores will make repeated preprocessing dramatically faster. Different data structures fit different access patterns:

- **Passages and processed training examples:** Apache Parquet/Arrow (or a Hugging Face Dataset backed by Arrow), partitioned into manageable shards. This gives column projection, memory mapping, and efficient streaming/shuffling.
- **Normalized alias lookup:** SQLite, DuckDB, or LMDB keyed by normalized alias, with one-to-many QID values. This avoids rescanning the 526 MB alias file for every corpus shard.
- **Knowledge graph adjacency:** dense integer node/relation IDs plus memory-mapped CSR-style arrays (`node_ptr`, `edge_dst`, `edge_type`, and direction). A graph is not naturally a row dataset; indexed adjacency is much faster for repeated neighborhood queries.
- **Reproducibility:** retain the original text files and record source hashes, normalization rules, tokenizer/version information, and preprocessing configuration in a manifest.

That conversion belongs in the later production preprocessing phase. This notebook intentionally keeps the logic transparent so we can first decide whether these examples and their graph paths are useful.